# Try FR3 Cartesian target control online

Edit Python, run a cell, and watch the simulated arm respond. The Python API publishes
Cartesian targets to the **native Rust controller**, while `franka-sim` runs alongside
this notebook inside your Binder session. You do not need to install software locally.

Choose **Run → Run All Cells** for the first experiment. Then change the parameters
or movement code and rerun those cells. The default motion lasts about 4.5 seconds,
plus startup and controller shutdown; startup may be slower on shared Binder hosts.

**This is a temporary playground.** Download the notebook to keep your edits before
closing the session. Binder may stop an idle session; files are not permanent.
Each experiment starts a fresh simulator with the same initial configuration and
stops it on exit. This lab uses the simulator on loopback, never a physical robot.


In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import franka
from sim_runtime import simulator
from sim_lab import (
    Parameters, measured_position, log_sample, live_view,
    plot_run, compare_runs, run_experiment,
)

if "runs" not in globals():
    runs = []
print("Ready. Run the parameter and motion cells below.")


## 1. Choose how the target should move

The target you request is a destination. The controller generates a smooth reference
on the way there; the simulated arm follows that reference.

| Setting | Meaning | Unit |
| --- | --- | --- |
| `max_velocity` | Speed limit for the generated translation | m/s |
| `max_acceleration` | Limit on how fast that velocity changes | m/s² |
| `max_jerk` | Limit on how fast acceleration changes | m/s³ |

Start with these defaults, then try `0.05`, `0.1`, and `1.0`. Change one setting at a
time to see its effect. A short move may never reach its velocity limit. With low
limits, the arm may still be moving when the next target is sent.


In [ ]:
# EDIT THESE VALUES, then rerun this cell and the motion cell.
max_velocity = 0.3       # m/s
max_acceleration = 0.5   # m/s²
max_jerk = 20.0          # m/s³
travel = 0.04           # 4 cm along x
hold_seconds = 1.5      # time between target updates

parameters = Parameters(
    max_velocity=max_velocity,
    max_acceleration=max_acceleration,
    max_jerk=max_jerk,
    amplitude=travel,
    hold_seconds=hold_seconds,
)
parameters.validate()


## 2. Edit and run the movement

`arm.move_by([dx, dy, dz])` offsets the **current target**, not the measured position.
`arm.move_to([x, y, z])` sets an absolute target. Distances are in metres.

This cell moves forward, backward past the starting point, then returns to the start.
Change the offsets or add another `arm.move_to(...)` followed by `observe(...)`.
`observe` only samples states and updates the viewer; the native controller continues
running between Python calls. The context managers wait for control to close and
stop the simulator before the cell finishes.

The viewer displays a kinematic skeleton: orange is the requested target; green is
the measured end-effector position. Its timeline also lets you replay the recording.


In [ ]:
if globals().get("active_task") is not None and not active_task.done():
    raise RuntimeError("Stop the slider experiment and wait before running this cell")

live_view()  # Attach the live viewer before logging motion.
rows = []

with simulator() as address:
    robot = arm = None
    try:
        robot = franka.Robot(address, realtime="ignore")
        model = robot.model()
        robot.set_collision_behavior_simple([40.] * 7, [40.] * 7, [40.] * 6, [40.] * 6)

        with robot.cartesian_targets(
            max_velocity=max_velocity,
            max_acceleration=max_acceleration,
            max_jerk=max_jerk,
            backend="impedance",
        ) as arm:
            start = np.array(arm.target()[:3], copy=True)
            wall_start = time.monotonic()

            def observe(seconds):
                deadline = time.monotonic() + seconds
                while time.monotonic() < deadline and arm.running:
                    state = arm.state()
                    if not rows or float(state.time) > rows[-1][0]:
                        q = np.array(state.q, copy=True)
                        target = np.array(arm.target()[:3], copy=True)
                        measured = measured_position(model, state, "impedance")
                        wire = np.array(state.O_T_EE[:3, 3], copy=True)
                        elapsed = time.monotonic() - wall_start
                        rows.append((float(state.time), elapsed, q, target, measured, wire))
                        if len(rows) % 2 == 0:
                            log_sample(model, elapsed, q, measured, target)
                    time.sleep(0.02)

            # YOUR MOVEMENT CODE: try changing these targets.
            arm.move_by([travel, 0.0, 0.0])
            observe(hold_seconds)
            arm.move_by([-2 * travel, 0.0, 0.0])
            observe(hold_seconds)
            arm.move_to(start)
            observe(hold_seconds)
    finally:
        # Drop native handles before stopping the simulator, including on errors.
        arm = None
        robot = None

# All native motion and simulator processes have stopped here.
if len(rows) < 4:
    raise RuntimeError("Too few samples; increase hold_seconds and run again.")
stamps, wall, q, target, measured, wire = map(np.asarray, zip(*rows))
result = dict(
    parameters=parameters, time=stamps - stamps[0], wall=wall,
    q=q, target=target, measured=measured, wire_measured=wire,
    measured_frame="FK EE from q and tool transforms", model=model,
)
runs.append(result)
print(f"Recorded {len(rows)} states. Controller closed; simulator stopped.")


## 3. Read the response

The dashed line is the requested destination, **not the smoothed controller
reference**. The measured position follows it with a delay. The impedance controller
also includes compliance and a following-distance leash, so doubling a limit does
not necessarily double the measured peak.

The derivative plots are approximate finite differences of sampled motion. They
help compare runs but do **not** certify the controller's exact 1 kHz velocity,
acceleration, or jerk limits. Jerk is especially sensitive to sampling noise.

**Coordinate frames:** this simulator's wire `O_T_EE` reports joint-7 rather than the
controller's end-effector frame. We use `model.pose("ee", state.q, state.F_T_EE,
state.EE_T_K)` for the impedance controller comparison and preserve the unmodified
wire positions in `result["wire_measured"]`.


In [ ]:
plot_run(result)
if len(runs) > 1:
    compare_runs(runs)


## 4. Try your own experiment

1. Change only `max_jerk` to `1.0`; rerun the parameter, motion, and plot cells.
2. Restore jerk and lower `max_velocity` to `0.05`. Does the arm finish each segment?
3. Replace a movement with `arm.move_to(start + [0.02, 0.01, 0.0])`.
4. Increase `hold_seconds` to give a slow trajectory more time to settle.

Each motion cell starts from a new simulation. `runs` retains the measured traces
inside this kernel; the comparison aligns each run's first measured position.
Keep the same target sequence and timing when comparing limits.

To save one trace, uncomment the following cell. Download the CSV through Jupyter's
file browser before the session ends.


In [ ]:
# np.savetxt(
#     "my-fr3-run.csv",
#     np.column_stack([result["time"], result["target"], result["measured"]]),
#     delimiter=",", header="time,target_x,target_y,target_z,measured_x,measured_y,measured_z",
#     comments="",
# )


## Optional: explore with sliders

These controls run the helper's fixed sequence: hold the start, move forward, move
backward, return to the start. Sliders affect the **next** run. Nothing starts merely
by executing this cell or using Run All. Click **Run experiment** to begin; **Stop**
requests orderly shutdown, which may take time to settle. Wait for completion before
running the teaching cells above again. Every slider experiment also gets a fresh simulator.


In [ ]:
import asyncio
import threading
import ipywidgets as widgets
from IPython.display import display, clear_output

if "active_task" not in globals():
    active_task = None

if active_task is not None and not active_task.done():
    raise RuntimeError("Stop the existing run and wait before recreating the controls")

# Retire the previous panel before rebinding callback globals on cell rerun.
for old_button, old_callback in globals().get("_panel_callbacks", []):
    old_button.on_click(old_callback, remove=True)
    old_button.disabled = True
for old_widget in globals().get("_panel_widgets", []):
    old_widget.close()
_panel_callbacks = []
_panel_widgets = []

velocity = widgets.FloatSlider(value=.3, min=.01, max=.8, step=.01, description="v (m/s)")
acceleration = widgets.FloatSlider(value=.5, min=.05, max=3., step=.05, description="a (m/s²)")
jerk = widgets.FloatSlider(value=20., min=.1, max=80., step=.1, description="j (m/s³)")
amplitude = widgets.FloatSlider(value=.04, min=.005, max=.08, step=.005, description="travel (m)")
hold = widgets.FloatSlider(value=1.5, min=.5, max=5., step=.5, description="hold (s)")
backend = widgets.Dropdown(options=["impedance", "robot"], value="impedance", description="backend")
run_button = widgets.Button(description="Run experiment", button_style="success")
stop_button = widgets.Button(description="Stop", button_style="warning", disabled=True)
status = widgets.HTML(value="Ready — no robot connection yet")
viewer_output, plot_output = widgets.Output(), widgets.Output()
controls = [velocity, acceleration, jerk, amplitude, hold, backend]
stop_event = threading.Event()

async def experiment(parameters):
    global active_task
    try:
        with viewer_output:
            clear_output(wait=True)
            live_view()  # Attach the notebook viewer BEFORE motion starts.
        worker = asyncio.create_task(asyncio.to_thread(run_experiment, parameters, stop_event, True))
        try:
            result = await asyncio.shield(worker)
        except asyncio.CancelledError:
            # Cancelling a Python task cannot cancel the native worker thread.
            # Request a stop and join before making the controls available again.
            stop_event.set()
            status.value = "Task cancelled; waiting for the controller to stop…"
            try:
                await asyncio.shield(worker)
            finally:
                status.value = "Cancelled; controller worker has finished"
            raise
        runs.append(result)
        missed = sum(not segment["reached"] for segment in result["segments"])
        with plot_output:
            clear_output(wait=True)
            plot_run(result)
        status.value = f"Run {len(runs)} {'stopped' if result['stopped'] else 'complete'}; controller closed; {missed} segments ended >5 mm from target"
    except Exception as error:
        # Do not retain exception objects/native robot references in notebook history.
        status.value = "Run failed; inspect the printed simulator error"
        with plot_output:
            print(type(error).__name__ + ": " + str(error))
    finally:
        for control in controls:
            control.disabled = False
        run_button.disabled, stop_button.disabled = False, True


def start(_):
    global active_task
    if active_task is not None and not active_task.done():
        return
    parameters = Parameters(velocity.value, acceleration.value, jerk.value,
                            amplitude.value, hold.value, backend=backend.value)
    parameters.validate()
    stop_event.clear()
    for control in controls:
        control.disabled = True
    run_button.disabled, stop_button.disabled = True, False
    status.value = "Running on the local simulator…"
    active_task = asyncio.create_task(experiment(parameters))


def stop(_):
    stop_event.set()
    status.value = "Stopping; waiting for the Rust controller to settle and close…"

run_button.on_click(start)
stop_button.on_click(stop)
display(widgets.VBox(controls + [widgets.HBox([run_button, stop_button]), status]),
        viewer_output, plot_output)

# Keep exact callback identities so an old panel cannot start a new run.
_panel_callbacks = [(run_button, start), (stop_button, stop)]
_panel_widgets = controls + [run_button, stop_button, status, viewer_output, plot_output]
